# Databricks/PySpark for Data Analysts/Engineers — Module 5: Spark SQL

So far we've written exclusively in the DataFrame API. This module shows Spark's other
face -- **Spark SQL**: exactly the same SQL queries you know from the SQL/PostgreSQL
course, run directly against our DataFrames.

## Table of contents
1. [createOrReplaceTempView and spark.sql()](#sec1)
2. [Mixing SQL and the DataFrame API](#sec2)
3. [Catalog: browsing registered views](#sec3)
4. [Global temp views](#sec4)
5. [Summary and exercises](#sec5)


We start (or grab the already-running) `SparkSession` -- the entry point to everything in Spark. In Databricks, `spark` is already available automatically in every notebook; the same code also works locally.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("spark_course").getOrCreate()
spark


Let's load our shared dataset -- online store data (`customers`, `products`, `employees`, `orders`, `order_items`), the exact same schema as the SQL course, but this time as Spark DataFrames.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- customers -----------------------------------------------------------
n_customers = 300
cities = ["Warsaw", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segments = ["New", "Standard", "Premium"]
segment_probs = [0.35, 0.45, 0.20]

customers_pd = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),
    "city": rng.choice(cities, size=n_customers, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segments, size=n_customers, p=segment_probs),
    "days_since_signup": rng.integers(1, 900, size=n_customers),
})

# --- products ------------------------------------------------------------
n_products = 40
categories = ["Electronics", "Clothing", "Home", "Sports", "Books"]
avg_prices = {"Electronics": 1200, "Clothing": 150, "Home": 250, "Sports": 300, "Books": 50}

product_categories = rng.choice(categories, size=n_products)
products_pd = pd.DataFrame({
    "product_id": np.arange(1, n_products + 1),
    "category": product_categories,
    "price": [
        round(max(5.0, rng.normal(avg_prices[c], avg_prices[c] * 0.3)), 2)
        for c in product_categories
    ],
})

# --- employees ----------------------------------------------------------
n_employees = 12
regions = ["North", "South", "East", "West"]
employees_pd = pd.DataFrame({
    "employee_id": np.arange(1, n_employees + 1),
    "region": rng.choice(regions, size=n_employees),
    "department": rng.choice(["Sales", "Support"], size=n_employees, p=[0.6, 0.4]),
})

# --- orders ------------------------------------------------------------
n_orders = 3000
order_customer_id = rng.choice(customers_pd["customer_id"], size=n_orders)
order_employee_id = rng.choice(employees_pd["employee_id"], size=n_orders)

segment_map = customers_pd.set_index("customer_id")["segment"]
order_segment = pd.Series(order_customer_id).map(segment_map).values

# cancellation probability depends on customer segment (deliberately built in)
cancel_prob = np.where(
    order_segment == "New", 0.22,
    np.where(order_segment == "Standard", 0.10, 0.04),
)
cancelled = rng.binomial(1, cancel_prob)
order_status = np.where(
    cancelled == 1, "Cancelled",
    rng.choice(["Placed", "Shipped", "Delivered"], size=n_orders, p=[0.2, 0.3, 0.5]),
)

base_date = pd.Timestamp("2024-01-01")
day_offset = rng.integers(0, 730, size=n_orders)

orders_pd = pd.DataFrame({
    "order_id": np.arange(1, n_orders + 1),
    "customer_id": order_customer_id,
    "employee_id": order_employee_id,
    "order_date": [base_date + pd.Timedelta(days=int(d)) for d in day_offset],
    "status": order_status,
})

# --- order_items ------------------------------------------------------
item_rows = []
item_id = 1
product_price = products_pd.set_index("product_id")["price"]

for order_id in orders_pd["order_id"]:
    n_items = rng.integers(1, 5)
    chosen_products = rng.choice(products_pd["product_id"], size=n_items, replace=False)
    for product_id in chosen_products:
        quantity = int(rng.integers(1, 4))
        discount = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        item_rows.append({
            "item_id": item_id,
            "order_id": int(order_id),
            "product_id": int(product_id),
            "quantity": quantity,
            "unit_price": float(product_price[product_id]),
            "discount": float(discount),
        })
        item_id += 1

order_items_pd = pd.DataFrame(item_rows)

# --- convert to Spark DataFrames ---------------------------------------
customers = spark.createDataFrame(customers_pd)
products = spark.createDataFrame(products_pd)
employees = spark.createDataFrame(employees_pd)
orders = spark.createDataFrame(orders_pd)
order_items = spark.createDataFrame(order_items_pd)

customers.show(5)


<a id="sec1"></a>
## 1. createOrReplaceTempView and spark.sql()

`createOrReplaceTempView("name")` registers a DataFrame as a **temporary SQL view**
(it only exists within the current `SparkSession`, and disappears once the notebook
closes). From that point on you can write ordinary SQL queries against it via
`spark.sql(...)`.

In [ ]:
customers.createOrReplaceTempView("customers_view")
products.createOrReplaceTempView("products_view")
orders.createOrReplaceTempView("orders_view")
order_items.createOrReplaceTempView("order_items_view")
employees.createOrReplaceTempView("employees_view")

spark.sql("SELECT * FROM customers_view LIMIT 5").show()


`spark.sql(...)` returns an ordinary DataFrame -- the exact same kind of object you'd get
from `select()`/`filter()`. You can call any DataFrame API method on it right away, or
assign it to a variable and keep using it further down in your code.

In [ ]:
result = spark.sql("""
    SELECT segment, COUNT(*) AS customer_count
    FROM customers_view
    GROUP BY segment
    ORDER BY customer_count DESC
""")
print(type(result))
result.show()


> ⚡ **The same SQL as in the SQL/PostgreSQL course**
>
> All the syntax learned in the SQL course -- `JOIN`, `GROUP BY`, `HAVING`, subqueries, `CASE WHEN`, window functions -- works here almost identically (Spark SQL is a dialect very close to the ANSI SQL standard, with small differences in some database-specific functions).

<a id="sec2"></a>
## 2. Mixing SQL and the DataFrame API

Since `spark.sql()` returns an ordinary DataFrame, you can freely MIX both styles in one
pipeline -- e.g. join tables in SQL and add further transformations in the DataFrame API,
or the other way around.

In [ ]:
# Join and aggregation in pure SQL
sql_report = spark.sql("""
    SELECT c.city, COUNT(*) AS order_count
    FROM orders_view o
    JOIN customers_view c ON o.customer_id = c.customer_id
    WHERE o.status != 'Cancelled'
    GROUP BY c.city
""")

# ...and further sorting and limiting -- in the DataFrame API
sql_report.orderBy(sql_report.order_count.desc()).limit(3).show()


In [ ]:
from pyspark.sql import functions as F

# The other way around: DataFrame API to prepare data...
filtered = orders.filter(F.col("status") == "Delivered")
filtered.createOrReplaceTempView("delivered_view")

# ...and then pure SQL again on the newly registered view
spark.sql("SELECT COUNT(*) AS delivered_count FROM delivered_view").show()


> ⚠️ **Choosing a style is about readability, not performance**
>
> Both SQL and the DataFrame API compile down to EXACTLY THE SAME execution plan via Catalyst -- neither one is faster. Pick whichever is more readable in a given spot: complex joins and aggregations are often more comfortable to write in SQL, while transformations driven by Python variables (loops, conditionals) fit the DataFrame API better.

<a id="sec3"></a>
## 3. Catalog: browsing registered views

`spark.catalog` gives you insight into what's currently registered in the running
session -- useful for checking which views/tables are available before writing a query
against them.

In [ ]:
for table in spark.catalog.listTables():
    print(table.name, "-- temporary:" if table.isTemporary else "-- permanent:", table.tableType)


In [ ]:
for column in spark.catalog.listColumns("customers_view"):
    print(column.name, column.dataType)


> 🧱 **Unity Catalog in Databricks**
>
> What we're seeing here (`spark.catalog`) is the simplest, local variant of a catalog -- temporary views that disappear once the session closes. In a real Databricks environment, **Unity Catalog** extends this with PERSISTENT tables, column- and row-level access permissions, and a shared catalog visible across the whole organization -- we come back to this in more detail in Module 14.

<a id="sec4"></a>
## 4. Global temp views

A plain `createOrReplaceTempView` is only visible within the CURRENT Spark session. If
you want to share a view across DIFFERENT sessions within the same Spark application,
use `createOrReplaceGlobalTempView` -- you then have to reference it through a special
`global_temp` database.

In [ ]:
products.createOrReplaceGlobalTempView("products_global")
spark.sql("SELECT * FROM global_temp.products_global LIMIT 3").show()


> ⚠️ **Global temp views in practice -- rarely needed**
>
> In typical work with a single notebook (as in this course), a plain `createOrReplaceTempView` is entirely sufficient. Global temp views are mainly useful in more complex applications with multiple independent `SparkSession` objects in one process -- a rare case in everyday analyst work.

<a id="sec5"></a>
## 5. Summary and exercises

In this module we covered:
- `createOrReplaceTempView()` -- registering a DataFrame as a temporary SQL view,
- `spark.sql()` -- running SQL queries that return an ordinary DataFrame,
- freely mixing SQL and DataFrame API style in one data pipeline,
- `spark.catalog` -- browsing registered views and their columns,
- `createOrReplaceGlobalTempView` -- views visible across sessions in the same application.

### 📝 Exercise 1: Your first SQL query

Register `employees` as the view `employees_view` and write a SQL query returning the number of employees in each department.

<details>
<summary>Show solution</summary>

```python
employees.createOrReplaceTempView("employees_view")
spark.sql("""
    SELECT department, COUNT(*) AS employee_count
    FROM employees_view
    GROUP BY department
""").show()
```

</details>

### 📝 Exercise 2: A join in SQL

Write a SQL query that joins `order_items_view` with `products_view` and computes the total revenue (`quantity * unit_price`) per category.

<details>
<summary>Show solution</summary>

```python
spark.sql("""
    SELECT p.category, ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue
    FROM order_items_view oi
    JOIN products_view p ON oi.product_id = p.product_id
    GROUP BY p.category
    ORDER BY revenue DESC
""").show()
```

</details>

### 📝 Exercise 3: SQL, then the DataFrame API

Run a SQL query returning `orders_view` filtered to status `'Cancelled'`, then call `.count()` on the RESULT (already a DataFrame).

<details>
<summary>Show solution</summary>

```python
cancelled = spark.sql("SELECT * FROM orders_view WHERE status = 'Cancelled'")
print(f"Number of cancelled orders: {cancelled.count()}")
```

</details>

### 📝 Exercise 4: Checking the catalog

Print, via `spark.catalog.listTables()`, the names of ALL views registered so far in this notebook.

<details>
<summary>Show solution</summary>

```python
for t in spark.catalog.listTables():
    print(t.name)
```

</details>

> 🔥 **Challenge: a subquery and CASE WHEN in Spark SQL**
>
> Write ONE SQL query (with a subquery, as in Module 6 of the SQL course) that: for each customer, sums the value of their orders (`quantity * unit_price * (1 - discount)`), then uses `CASE WHEN` to categorize customers into `'VIP'` (sum > 5000) and `'Regular'` -- count how many customers fall into each category.

<details>
<summary>Show solution</summary>

```python
spark.sql("""
    SELECT
        CASE WHEN total_value > 5000 THEN 'VIP' ELSE 'Regular' END AS category,
        COUNT(*) AS customer_count
    FROM (
        SELECT
            o.customer_id,
            SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) AS total_value
        FROM orders_view o
        JOIN order_items_view oi ON o.order_id = oi.order_id
        GROUP BY o.customer_id
    ) AS total_per_customer
    GROUP BY category
""").show()
```

This is exactly the same FROM-clause subquery pattern from Module 6 of the SQL course -- it works identically in Spark SQL.

</details>

## What's next?

In **Module 6** we'll cover reading and writing data: CSV/JSON/Parquet files, explicit vs.
inferred schemas, and partitioning on write.